# LC11 — Trees, forests and boosting: how the winning models work (self-paced, ~45 min)

LC10's gradient boosting quietly won the forecast table. Before anyone uses a model that wins, they should know **how** it wins and **how** it fails. This notebook opens the box: one decision tree, its overfitting disease, and the two cures — averaging (forests) and correcting (boosting). Material appears in **Quiz 4**; the second half of the Lecturecise 11 session belongs to our industry guest.

In [ ]:
# Install exactly what this notebook uses.
%pip install scikit-learn pandas numpy pyarrow matplotlib --quiet
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
df = pd.read_parquet("../data/svedala-year/svedala_hourly.parquet")
y = df["ZON_MITT"].interpolate(limit=3)
X = pd.DataFrame(index=df.index)
X["lag24"], X["lag168"] = y.shift(24), y.shift(168)
X["temp"] = df["temp_mid"].interpolate(limit=3)
X["hour_sin"] = np.sin(2*np.pi*df.index.hour/24); X["hour_cos"] = np.cos(2*np.pi*df.index.hour/24)
X["dow"] = df.index.dayofweek; X["workday"] = (X["dow"] < 5).astype(int)
data = X.join(y.rename("target")).dropna()
train = data.loc[:"2025-09-30"]; test = data.loc["2025-10-01":]
Xtr, ytr = train.drop(columns="target"), train["target"]
Xte, yte = test.drop(columns="target"), test["target"]
print(f"features: {list(Xtr.columns)}")

In [ ]:
from pathlib import Path
# Guard: this notebook expects to run from the notebooks/ folder of a clone of
# the course repository — the datasets live one level up in ../data/.
# Failing here, early and clearly, beats a confusing FileNotFoundError later.
assert Path("../data").exists(), (
    "Course data folder not found. Clone KTH-EG2140/course-material and open "
    "this notebook from its notebooks/ folder.")

## 1. One tree — readable, and that is the point

A depth-3 tree is a flowchart a human can audit. Read the splits: the tree *rediscovers* what you know about load — yesterday's value first, then temperature, then time of week:

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree
import matplotlib.pyplot as plt
t3 = DecisionTreeRegressor(max_depth=3, random_state=0).fit(Xtr, ytr)
plt.figure(figsize=(13, 4)); plot_tree(t3, feature_names=list(Xtr.columns), fontsize=7, filled=False);

## 2. The disease: a tree memorizes

Let the depth grow and watch train error fall toward zero while test error turns around — the textbook overfitting curve, on real data:

In [ ]:
# The overfitting experiment: same data, growing depth (None = unlimited).
# For each depth, record TRAIN error (falls forever) and TEST error (turns).
depths = [2, 3, 4, 6, 8, 12, 16, None]
rows = []
for d in depths:
    m = DecisionTreeRegressor(max_depth=d, random_state=0).fit(Xtr, ytr)
    rows.append({"depth": str(d), "train_MAE": np.abs(m.predict(Xtr)-ytr).mean(),
                 "test_MAE": np.abs(m.predict(Xte)-yte).mean()})
res = pd.DataFrame(rows).set_index("depth").round(1)
print(res.to_string())
res.plot(figsize=(7,3), title="Deeper is better — until it very much is not");

## 3. Two cures

**Forest** (bagging): train many deep trees on random subsets, average away their individual nonsense. **Boosting**: train shallow trees *in sequence*, each one fitted to the errors of the sum so far. Different philosophies, both tame the variance:

In [ ]:
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
comp = {"single tree (best depth)": res["test_MAE"].astype(float).min()}
rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=0).fit(Xtr, ytr)
comp["random forest"] = np.abs(rf.predict(Xte)-yte).mean()
gb = HistGradientBoostingRegressor(random_state=0).fit(Xtr, ytr)
comp["gradient boosting"] = np.abs(gb.predict(Xte)-yte).mean()
print(pd.Series(comp).round(1).to_string())

## 4. Feature importance — useful, and routinely over-read

Every applied ML talk shows an importance bar chart. Two caveats before you ever put one in a report: importances split credit arbitrarily between **correlated features** (lag24 and lag168 share the same signal), and importance is **not causality** — temperature "mattering" does not tell an operator what happens if they could change it:

In [ ]:
imp = pd.Series(rf.feature_importances_, index=Xtr.columns).sort_values()
imp.plot.barh(figsize=(6,2.5), title="RF importance — read with both caveats in mind");
print(imp.round(3).to_string())

## Self-check

In [ ]:
best_single = res["test_MAE"].astype(float).min()
assert comp["gradient boosting"] < best_single, "boosting should beat any single tree"
assert res["train_MAE"].astype(float).iloc[-1] < 5, "unlimited tree should near-memorize the train set"
print("ALL OK — you know how the winner wins. Lab 8 points the same models at N-1 security.")